# 02 — Data Augmentation
**Objectif :** Augmenter artificiellement le dataset pour :
- Compenser le manque de données (253 images c'est trop peu)
- Corriger le déséquilibre entre classes (155 yes vs 98 no)

**Résultat attendu :** 253 → 2065 images

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 1. Définir les transformations d'augmentation

In [ ]:
def augment_image(img):
    """
    Applique 7 transformations à une image et retourne une liste de 8 images
    (l'originale + 7 variantes).
    """
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    augmented = [img]

    # 1. Miroir horizontal
    augmented.append(cv2.flip(img, 1))

    # 2. Rotation +10°
    M = cv2.getRotationMatrix2D(center, 10, 1)
    augmented.append(cv2.warpAffine(img, M, (w, h)))

    # 3. Rotation -10°
    M = cv2.getRotationMatrix2D(center, -10, 1)
    augmented.append(cv2.warpAffine(img, M, (w, h)))

    # 4. Zoom léger (crop centre + resize)
    zoom = 0.85
    r, c = int(h * zoom / 2), int(w * zoom / 2)
    cy, cx = h // 2, w // 2
    cropped = img[cy-r:cy+r, cx-c:cx+c]
    augmented.append(cv2.resize(cropped, (w, h)))

    # 5. Luminosité augmentée
    augmented.append(cv2.convertScaleAbs(img, alpha=1.2, beta=20))

    # 6. Luminosité réduite
    augmented.append(cv2.convertScaleAbs(img, alpha=0.8, beta=-20))

    # 7. Miroir + rotation
    flipped = cv2.flip(img, 1)
    M = cv2.getRotationMatrix2D(center, 15, 1)
    augmented.append(cv2.warpAffine(flipped, M, (w, h)))

    return augmented  # 8 images au total

## 2. Visualiser les transformations sur une image

In [ ]:
sample_path = os.path.join('data', 'yes', os.listdir('data/yes')[0])
img = cv2.imread(sample_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

variants = augment_image(img)
labels = ['Original', 'Miroir', 'Rot +10°', 'Rot -10°', 'Zoom', 'Bright +', 'Bright -', 'Miroir+Rot']

fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for i, (variant, label) in enumerate(zip(variants, labels)):
    axes[i].imshow(variant)
    axes[i].set_title(label, fontsize=8)
    axes[i].axis('off')

plt.suptitle('Les 8 variantes générées par image', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Générer et sauvegarder les images augmentées

In [ ]:
def augment_class(src_folder, dest_folder, n_generated, label):
    """
    Pour chaque image dans src_folder, génère des variantes
    jusqu'à atteindre n_generated images dans dest_folder.
    """
    os.makedirs(dest_folder, exist_ok=True)

    files = [f for f in os.listdir(src_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    count = 0

    for f in files:
        path = os.path.join(src_folder, f)
        img = cv2.imread(path)
        if img is None:
            continue

        variants = augment_image(img)

        for i, variant in enumerate(variants):
            if count >= n_generated:
                break
            fname = f'{label}_{count}.jpg'
            cv2.imwrite(os.path.join(dest_folder, fname), variant)
            count += 1

        if count >= n_generated:
            break

    print(f'[{label}] {len(files)} originales → {count} images générées')

# yes → 1085 images, no → 980 images
augment_class('data/yes', 'data/augmented_data/yes', n_generated=1085, label='yes')
augment_class('data/no',  'data/augmented_data/no',  n_generated=980,  label='no')

## 4. Vérifier le résultat

In [ ]:
yes_aug = len(os.listdir('data/augmented_data/yes'))
no_aug  = len(os.listdir('data/augmented_data/no'))

print(f'yes (augmenté) : {yes_aug}')
print(f'no  (augmenté) : {no_aug}')
print(f'Total          : {yes_aug + no_aug}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['yes (tumeur)', 'no (sain)'], [yes_aug, no_aug], color=['tomato', 'steelblue'])
ax.set_title('Distribution après augmentation')
ax.set_ylabel("Nombre d'images")
for i, v in enumerate([yes_aug, no_aug]):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()